# NEGF Time Evolution GUI

This notebook implements an interactive GUI for handling HDF5 files, visualizing parameters, setting up spatial masks, selecting quasiparticles, configuring laser and VCAP parameters, and running time evolution simulations.

## 1. File Operations

Specify the path and name of the HDF5 file, then load the parameters.

In [1]:
# Required imports and file tab
import ipywidgets as widgets
from IPython.display import display, clear_output
import h5py
import os
import glob
import numpy as np

file_output = widgets.Output()

directory_widget = widgets.Text(
    value=os.getcwd(),
    description="Directory:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
filename_widget = widgets.Text(
    value="scatter_data.h5",
    description="Filename:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

def get_hdf5_path():
    return os.path.join(directory_widget.value, filename_widget.value)

file_load_button = widgets.Button(description="Load", button_style='success')

loaded_params = {}

def load_hdf5_file(b):
    with file_output:
        clear_output(wait=True)
        path = get_hdf5_path()
        if not os.path.exists(path):
            print("The specified file was not found.")
            return
        try:
            with h5py.File(path, "r") as f:
                params = {}
                for k, v in f.attrs.items():
                    params[k] = v
                if "params" in f:
                    for k, v in f["params"].attrs.items():
                        params[k] = v
                loaded_params.clear()
                loaded_params.update(params)
                print(f"Successfully loaded: {path}")
        except Exception as e:
            print(f"Error while reading file: {e}")
    # Refresh parameter tab after loading
    update_param_tab()

file_load_button.on_click(load_hdf5_file)

file_tab = widgets.VBox([
    widgets.Label("Specify HDF5 file path and name"),
    directory_widget,
    filename_widget,
    file_load_button,
    file_output
])

## 2. Parameters and Visualization

Display the loaded parameters in a structured way, and show graphical representations of the potential and dispersion.

In [2]:
# Parameters tab: structured display and plots
import matplotlib.pyplot as plt

def create_param_widgets():
    grid_keys = ['uxgrid_num', 'uxgrid_dx', 'uxgrid_width']
    unit_keys = ['hbar', 'm0', 'kappa0', 'e0']
    pot_keys = ['wallwidth', 'wallheight', 'wallrise', 'wellwidth', 'welldepth', 'wellfall', 'num_cells', 'cell_spacing']
    disp_keys = ['disp_grid_mode', 'disp_energy_min', 'disp_energy_max', 'disp_number_of_points']

    grid_items, unit_items, pot_items, disp_items = [], [], [], []

    for k, v in loaded_params.items():
        if k in grid_keys:
            grid_items.append(widgets.Text(value=str(v), description=f"{k}:", disabled=True, style={'description_width': 'initial'}))
        elif k in unit_keys:
            unit_items.append(widgets.Text(value=str(v), description=f"{k}:", disabled=True, style={'description_width': 'initial'}))
        elif k in pot_keys:
            continue
        elif k in disp_keys:
            disp_items.append(widgets.Text(value=str(v), description=f"{k}:", disabled=True, style={'description_width': 'initial'}))

    # Potential plot
    pot_plot_output = widgets.Output()
    with pot_plot_output:
        pot_plot_output.clear_output(wait=True)
        try:
            import numpy as np
            wallwidth = float(loaded_params.get('wallwidth', 3.0))
            wallheight = float(loaded_params.get('wallheight', 1.0))
            wallrise = float(loaded_params.get('wallrise', 1.0))
            wellwidth = float(loaded_params.get('wellwidth', 10.0))
            welldepth = float(loaded_params.get('welldepth', 1.0))
            wellfall = float(loaded_params.get('wellfall', 1.0))
            num_cells = int(loaded_params.get('num_cells', 1))
            cell_spacing = float(loaded_params.get('cell_spacing', 0.0))
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
            pot = np.zeros_like(uxgrid)
            cell_length = 2*wallwidth + wellwidth + cell_spacing
            for i in range(num_cells):
                offset = (i - (num_cells-1)/2) * cell_length
                left = offset - (wellwidth/2 + wallwidth)
                right = offset + (wellwidth/2 + wallwidth)
                well_left = offset - wellwidth/2
                well_right = offset + wellwidth/2
                pot[(uxgrid >= left) & (uxgrid < well_left)] = wallheight
                pot[(uxgrid > well_right) & (uxgrid <= right)] = wallheight
                pot[(uxgrid >= well_left) & (uxgrid <= well_right)] = -welldepth
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, pot)
            plt.xlabel("x")
            plt.ylabel("Potential")
            plt.title("Potential (simplified)")
            plt.grid()
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("Potential plot error:", e)

    # Dispersion plot
    disp_plot_output = widgets.Output()
    with disp_plot_output:
        disp_plot_output.clear_output(wait=True)
        try:
            disp_mode = loaded_params.get('disp_grid_mode', 'unified in k')
            e_min = float(loaded_params.get('disp_energy_min', 0.05))
            e_max = float(loaded_params.get('disp_energy_max', 2.0))
            npts = int(loaded_params.get('disp_number_of_points', 10))
            hbar = float(loaded_params.get('hbar', 1.0))
            m0 = float(loaded_params.get('m0', 8.8))
            e_vals = np.linspace(e_min, e_max, npts)
            k_vals = np.sqrt(2*m0*e_vals)/hbar
            plt.figure(figsize=(6,2.5))
            if disp_mode == 'unified in k':
                plt.plot(k_vals, e_vals, 'o-')
                plt.xlabel("k")
                plt.ylabel("energy")
                plt.title("Dispersion (simplified, by k)")
            else:
                plt.plot(e_vals, k_vals, 'o-')
                plt.xlabel("energy")
                plt.ylabel("k")
                plt.title("Dispersion (simplified, by energy)")
            plt.grid()
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("Dispersion plot error:", e)

    sections = []
    if grid_items:
        sections.append(widgets.HTML("<b>Grid parameters</b>"))
        sections.append(widgets.VBox(grid_items))
    if unit_items:
        sections.append(widgets.HTML("<b>Unit system parameters</b>"))
        sections.append(widgets.VBox(unit_items))
    sections.append(widgets.HTML("<b>Potential (plot)</b>"))
    sections.append(pot_plot_output)
    if disp_items:
        sections.append(widgets.HTML("<b>Dispersion parameters</b>"))
        sections.append(widgets.VBox(disp_items))
        sections.append(disp_plot_output)
    if not sections:
        sections.append(widgets.Label("No parameters loaded."))
    return widgets.VBox(sections)

param_tab_output = widgets.Output()

def update_param_tab(*args):
    with param_tab_output:
        clear_output(wait=True)
        display(create_param_widgets())

# Refresh parameter tab after file load
file_load_button.on_click(lambda b: update_param_tab())

param_tab = widgets.VBox([
    widgets.Label("Loaded parameters (structured, with plots):"),
    param_tab_output
])
update_param_tab()

## 3. Quasi Particles

List quasiparticles read from the HDF5 file, with multi-selection support.

In [3]:
# Quasi Particles tab
quasiparticle_list = []
quasiparticle_labels = []
selected_quasiparticles = []

def extract_quasiparticles_from_hdf5():
    global quasiparticle_list, quasiparticle_labels
    quasiparticle_list = []
    quasiparticle_labels = []
    path = get_hdf5_path()
    if not os.path.exists(path):
        return
    try:
        import quantum_toolkit as quat
        with h5py.File(path, "r") as f:
            if "scatter" in f:
                grp = f["scatter"]
                if "quasiparticles" in grp:
                    qp_group = grp["quasiparticles"]
                    for i in range(len(qp_group)):
                        qpg = qp_group[str(i)]
                        qp_dict = {}
                        for key in qpg:
                            if isinstance(qpg[key], h5py.Group):
                                subg = qpg[key]
                                subdict = {}
                                for sk in subg:
                                    subdict[sk] = subg[sk][()]
                                for sk in subg.attrs:
                                    subdict[sk] = subg.attrs[sk]
                                qp_dict[key] = subdict
                            else:
                                qp_dict[key] = qpg[key][()]
                        for key in qpg.attrs:
                            qp_dict[key] = qpg.attrs[key]
                        # Ensure all Particle fields are loaded
                        # Fallbacks for missing fields
                        qp_dict.setdefault("charge", loaded_params.get("e0", 1.0))
                        qp_dict.setdefault("mass", loaded_params.get("m0", 1.0))
                        qp_dict.setdefault("wavenumber", qp_dict.get("k", None))
                        qp_dict.setdefault("angular_frequency", qp_dict.get("omega", 0.0))
                        qp_dict.setdefault("state", qp_dict.get("state", None))
                        # Convert to QuasiParticle object if available, else dict
                        try:
                            qp_obj = quat.QuasiParticle(**qp_dict)
                        except Exception:
                            qp_obj = qp_dict
                        quasiparticle_list.append(qp_obj)
                        try:
                            label = f"#{i} | E={qp_dict.get('energy','?')} | k={qp_dict.get('k', qp_dict.get('wavenumber','?'))} | T={qp_dict.get('transmission','?')} | R={qp_dict.get('reflection','?')}"
                        except Exception:
                            label = f"#{i}"
                        quasiparticle_labels.append(label)
                else:
                    # Old format: just arrays
                    k_vals = grp["k_vals"][:] if "k_vals" in grp else []
                    e_vals = grp["e_vals"][:] if "e_vals" in grp else []
                    t_vals = grp["t_vals"][:] if "t_vals" in grp else []
                    r_vals = grp["r_vals"][:] if "r_vals" in grp else []
                    wf = grp["wavefunction"][:] if "wavefunction" in grp else None
                    for i in range(len(k_vals)):
                        qp_kwargs = {
                            "index": i,
                            "k": k_vals[i],
                            "energy": e_vals[i],
                            "transmission": t_vals[i],
                            "reflection": r_vals[i],
                            "wavefunction": wf[i] if wf is not None and len(wf.shape) > 1 else None,
                            "charge": loaded_params.get("e0", 1.0),
                            "mass": loaded_params.get("m0", 1.0),
                            "wavenumber": k_vals[i],
                            "angular_frequency": 0.0,
                            "state": {"grid": loaded_params.get("uxgrid", None), "value": wf[i]} if wf is not None and len(wf.shape) > 1 else None
                        }
                        try:
                            qp_obj = quat.QuasiParticle(**qp_kwargs)
                        except Exception:
                            qp_obj = qp_kwargs
                        quasiparticle_list.append(qp_obj)
                        label = f"#{i} | E={e_vals[i]} | k={k_vals[i]} | T={t_vals[i]} | R={r_vals[i]}"
                        quasiparticle_labels.append(label)
    except Exception as e:
        pass

def update_quasiparticle_tab(*args):
    extract_quasiparticles_from_hdf5()
    with quasiparticle_tab_output:
        clear_output(wait=True)
        if not quasiparticle_list:
            display(widgets.Label("No quasiparticle data found in the file."))
            return
        # Multi-selection
        select_all_checkbox = widgets.Checkbox(value=False, description="Select all")
        multi_select = widgets.SelectMultiple(
            options=[(label, i) for i, label in enumerate(quasiparticle_labels)],
            value=(),
            description="Quasiparticles:",
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='600px', height='200px')
        )
        info_output = widgets.Output()

        def on_select_all_change(change):
            if change["new"]:
                multi_select.value = tuple(range(len(quasiparticle_list)))
            else:
                multi_select.value = ()

        def on_multi_select_change(change):
            global selected_quasiparticles
            selected_quasiparticles = [quasiparticle_list[i] for i in multi_select.value]
            with info_output:
                clear_output(wait=True)
                if not selected_quasiparticles:
                    print("No quasiparticle selected.")
                else:
                    print(f"Selected {len(selected_quasiparticles)} quasiparticle(s):")
                    for idx, qp in enumerate(selected_quasiparticles):
                        # Try attribute access, fallback to dict-style if needed
                        try:
                            index = getattr(qp, 'index', None)
                            if index is None:
                                index = getattr(qp, 'idx', None)
                            if index is None:
                                index = idx
                            energy = getattr(qp, 'energy', qp.get('energy', '?'))
                            k = getattr(qp, 'k', getattr(qp, 'wavenumber', qp.get('k', '?')))
                            transmission = getattr(qp, 'transmission', qp.get('transmission', '?'))
                            reflection = getattr(qp, 'reflection', qp.get('reflection', '?'))
                        except Exception:
                            # fallback to dict-style
                            index = qp.get('index', idx)
                            energy = qp.get('energy', '?')
                            k = qp.get('k', '?')
                            transmission = qp.get('transmission', '?')
                            reflection = qp.get('reflection', '?')
                        # Safe formatting: only format as float if possible, else print as string
                        def fmt(val, fmtstr):
                            try:
                                return format(float(val), fmtstr)
                            except Exception:
                                return str(val)
                        print(
                            f"  index={index}  E={fmt(energy, '.3e')}  k={fmt(k, '.3e')}  T={fmt(transmission, '.2f')}  R={fmt(reflection, '.2f')}"
                        )

        select_all_checkbox.observe(on_select_all_change, names="value")
        multi_select.observe(on_multi_select_change, names="value")
        # Default: nothing selected
        selected_quasiparticles.clear()
        display(widgets.VBox([
            widgets.Label("Quasiparticle list (multi-selection possible):"),
            select_all_checkbox,
            multi_select,
            info_output
        ]))

quasiparticle_tab_output = widgets.Output()
update_quasiparticle_tab()

# Refresh quasiparticle tab after file load
file_load_button.on_click(lambda b: update_quasiparticle_tab())

quasiparticle_tab = widgets.VBox([
    widgets.Label("Quasi Particles (multi-selection, for later use):"),
    quasiparticle_tab_output
])

## 4. Mask Setup

Select spatial mask ranges and points, with visualization.

In [4]:
# Mask tab: spatial mask setup and visualization
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Default parameters
default_uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
default_uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
default_uxgrid_width = default_uxgrid_num * default_uxgrid_dx
default_uxgrid = np.arange(-default_uxgrid_width/2, default_uxgrid_width/2, default_uxgrid_dx)

# Mask parameter widgets
mask_enable_checkbox = widgets.Checkbox(value=False, description="Enable mask")
mask_ranges_text = widgets.Text(
    value="-121,-119;119,121",
    description="Highlighted range(s):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)
mask_points_widget = widgets.IntText(
    value=1024,
    description="Number of points (reduced):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
mask_plot_output = widgets.Output()

# Global mask variable
current_mask = np.ones(default_uxgrid_num, dtype=bool)

def parse_ranges(ranges_str):
    """E.g. '-121,-119;119,121' -> [(-121, -119), (119, 121)]"""
    try:
        ranges = []
        for part in ranges_str.split(';'):
            if not part.strip():
                continue
            start, end = map(float, part.strip().split(','))
            ranges.append((start, end))
        return ranges
    except Exception:
        return []

def update_mask_plot(*args):
    global current_mask
    uxgrid_num = int(loaded_params.get('uxgrid_num', default_uxgrid_num))
    uxgrid_dx = float(loaded_params.get('uxgrid_dx', default_uxgrid_dx))
    uxgrid_width = uxgrid_num * uxgrid_dx
    uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
    desired_points = mask_points_widget.value
    step_size = max(1, uxgrid_num // max(1, desired_points))
    mask = np.zeros(uxgrid_num, dtype=bool)
    mask[::step_size] = True

    highlight_ranges = parse_ranges(mask_ranges_text.value)
    mask_highlight_ranges = np.zeros(uxgrid_num, dtype=bool)
    for start, end in highlight_ranges:
        mask_highlight_ranges |= (uxgrid >= start) & (uxgrid <= end)
    mask = mask | mask_highlight_ranges

    # Enable/disable mask
    if mask_enable_checkbox.value:
        current_mask = mask
    else:
        current_mask = np.ones(uxgrid_num, dtype=bool)

    with mask_plot_output:
        clear_output(wait=True)
        plt.figure(figsize=(7,2.5))
        plt.plot(uxgrid, np.zeros_like(uxgrid), 'o', markersize=2, label='Original uxgrid')
        plt.plot(uxgrid[current_mask], np.zeros_like(uxgrid[current_mask]), 'o', markersize=4, label='Active mask')
        for start, end in highlight_ranges:
            plt.axvspan(start, end, color='orange', alpha=0.2)
        plt.xlabel('uxgrid')
        plt.ylabel('Value')
        plt.title('Mask visualization')
        plt.legend()
        plt.show()
        print(f"Number of masked points: {np.sum(current_mask)} / {uxgrid_num}")

# Widget event handlers
mask_enable_checkbox.observe(update_mask_plot, names='value')
mask_ranges_text.observe(update_mask_plot, names='value')
mask_points_widget.observe(update_mask_plot, names='value')

# Initial update
update_mask_plot()

mask_tab = widgets.VBox([
    widgets.Label("Spatial mask setup:"),
    mask_enable_checkbox,
    widgets.HBox([mask_points_widget, mask_ranges_text]),
    mask_plot_output
])

## 5. Laser Field Configuration

Set external laser field parameters and visualize them.

In [5]:
# Laser tab: parameter setup and visualization
import matplotlib.pyplot as plt
from quantum_toolkit import potentials as pots

# Default values
default_spot_size_nm = 800
default_field_strength_gvm = 1.0
default_central_wavelength_nm = 800
default_num_of_cycles = 5
default_beta = 0.2

laser_spot_size_widget = widgets.FloatText(
    value=default_spot_size_nm,
    description="Spot size [nm]:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_field_strength_widget = widgets.FloatText(
    value=default_field_strength_gvm,
    description="Field strength [GV/m]:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_wavelength_widget = widgets.FloatText(
    value=default_central_wavelength_nm,
    description="Wavelength [nm]:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_cycles_widget = widgets.IntText(
    value=default_num_of_cycles,
    description="Number of cycles:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)
laser_beta_widget = widgets.FloatText(
    value=default_beta,
    description="Beta:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px')
)

laser_plot_output = widgets.Output()

def update_laser_plot(*args):
    with laser_plot_output:
        clear_output(wait=True)
        try:
            # Parameters
            spot_size_nm = laser_spot_size_widget.value
            field_strength_gvm = laser_field_strength_widget.value
            central_wavelength_nm = laser_wavelength_widget.value
            num_of_cycles = laser_cycles_widget.value
            beta = laser_beta_widget.value

            # Grid parameters
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)

            # Unit system (if available)
            try:
                import atomic_units as au
                hb = float(loaded_params.get('hbar', 1.0))
                m0 = float(loaded_params.get('m0', 8.8))
                kappa0 = float(loaded_params.get('kappa0', 2.3))
                e0 = float(loaded_params.get('e0', 3.4))
                unit_sys = au.AtomicUnitSystem(xh=hb, xe=e0, xm=m0, xk=kappa0)
            except Exception:
                unit_sys = None

            # Laser parameter conversion
            if unit_sys is not None:
                central_wavelength = central_wavelength_nm * unit_sys.convert_length_from("nm")
                s_min = -0.5 * spot_size_nm * unit_sys.convert_length_from("nm")
                s_max = 0.5 * spot_size_nm * unit_sys.convert_length_from("nm")
                magnitude = field_strength_gvm * unit_sys.convert_electric_field_from("GV/m")
                omega = 2 * np.pi * unit_sys.speed_of_light / central_wavelength_nm
                t_min = 0
                t_max = num_of_cycles * 2 * np.pi / omega
                utgrid_dt = 0.05
                utgrid = np.arange(t_min, t_max, utgrid_dt)
                uxgrid_nm = uxgrid * unit_sys.length_unit.to('nm').magnitude
                tvals_fs = utgrid * unit_sys.time_unit.to('fs').magnitude
                field_unit_gvm = unit_sys.electric_field_unit.to('GV/m').magnitude
            else:
                s_min = -0.5 * spot_size_nm
                s_max = 0.5 * spot_size_nm
                magnitude = field_strength_gvm
                t_min = 0
                t_max = num_of_cycles * 2 * np.pi / (3e8 / (central_wavelength_nm * 1e-9))
                utgrid_dt = 0.05
                utgrid = np.arange(t_min, t_max, utgrid_dt)
                uxgrid_nm = uxgrid
                tvals_fs = utgrid
                field_unit_gvm = 1.0

            # Laser field object
            try:
                laserpot = pots.SmoothLaserPotential(
                    uxgrid,
                    spatial_min=s_min, spatial_max=s_max,
                    temporal_max=t_max, temporal_min=t_min,
                    noc=num_of_cycles,
                    amplitude=magnitude,
                    beta=beta
                )
            except Exception as e:
                print("Error creating laser field object:", e)
                return

            # Plot: spatial profile at a given time
            plt.figure(figsize=(6,2.5))
            plt.plot(
                uxgrid_nm,
                laserpot(laserpot.temporal_width/2) * field_unit_gvm,
                label="Laser (spatial profile)"
            )
            plt.xlabel("x [nm]")
            plt.ylabel("Field [GV/m]")
            plt.title("Laser field spatial profile (mid time)")
            plt.grid()
            plt.legend()
            plt.tight_layout()
            plt.show()

            # Plot: temporal profile at x=0
            plt.figure(figsize=(6,2.5))
            x0_idx = np.abs(uxgrid).argmin()
            yvals = [laserpot.value_at(uxgrid[x0_idx], t) * field_unit_gvm for t in utgrid]
            plt.plot(tvals_fs, yvals, label="Laser (temporal profile, x=0)")
            plt.xlabel("time [fs]")
            plt.ylabel("Field [GV/m]")
            plt.title("Laser field temporal profile (x=0)")
            plt.grid()
            plt.legend()
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("Laser plot error:", e)

for w in [laser_spot_size_widget, laser_field_strength_widget, laser_wavelength_widget, laser_cycles_widget, laser_beta_widget]:
    w.observe(update_laser_plot, names='value')

update_laser_plot()

laser_tab = widgets.VBox([
    widgets.Label("External laser field parameters:"),
    widgets.HBox([laser_spot_size_widget, laser_field_strength_widget]),
    widgets.HBox([laser_wavelength_widget, laser_cycles_widget, laser_beta_widget]),
    laser_plot_output
])

## 6. VCAP Parameters

Set VCAP parameters and visualize them automatically.

In [6]:
# VCAP tab: parameters and visualization
import ipywidgets as widgets
from IPython.display import display, clear_output

# VCAP parameter widgets
vcap_x0_widget = widgets.FloatText(
    value=350.0,
    description="param_x0:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
vcap_lambda0_widget = widgets.FloatText(
    value=0.05,
    description="param_lambda0:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
vcap_plot_output = widgets.Output()

def update_vcap_plots(*args):
    with vcap_plot_output:
        clear_output(wait=True)
        try:
            import matplotlib.pyplot as plt
            import numpy as np
            import quantum_toolkit as quat
            # uxgrid parameters
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
            # parameters
            param_x0 = vcap_x0_widget.value
            param_lambda0 = vcap_lambda0_widget.value
            v_cap = quat.vcap_generator(uxgrid, param_x0=param_x0, param_lambda0=param_lambda0)
            # 1. plot: abs and angle
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, np.abs(v_cap[0]), label="|Vcap[0]|")
            plt.plot(uxgrid, np.angle(v_cap[0]), label="Angle{Vcap[0]}")
            plt.legend()
            plt.grid()
            plt.title("VCAP[0]: abs and angle")
            plt.tight_layout()
            plt.show()
            # 2. plot: Re/Im v_cap[1]
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, np.real(v_cap[1]), label="Re{Vcap[1]}")
            plt.plot(uxgrid, np.imag(v_cap[1]), label="Im{Vcap[1]}")
            plt.legend()
            plt.grid()
            plt.title("VCAP[1]: Re and Im")
            plt.tight_layout()
            plt.show()
            # 3. plot: Re/Im v_cap[2]
            plt.figure(figsize=(6,2.5))
            plt.plot(uxgrid, np.real(v_cap[2]), label="Re{Vcap[2]}")
            plt.plot(uxgrid, np.imag(v_cap[2]), label="Im{Vcap[2]}")
            plt.legend()
            plt.grid()
            plt.title("VCAP[2]: Re and Im")
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("VCAP plot error:", e)

# Auto update on parameter change
vcap_x0_widget.observe(update_vcap_plots, names='value')
vcap_lambda0_widget.observe(update_vcap_plots, names='value')

update_vcap_plots()

vcap_tab = widgets.VBox([
    widgets.Label("VCAP parameters and visualization:"),
    widgets.HBox([vcap_x0_widget, vcap_lambda0_widget]),
    vcap_plot_output
])

## 7. Time Evolution (Compute)

Run time evolution for the selected quasiparticles, save results and visualize them.

In [7]:
# Compute tab: time evolution for selected quasiparticles
import ipywidgets as widgets
from IPython.display import display, clear_output
import h5py
import numpy as np
import matplotlib.pyplot as plt
import quantum_toolkit as quat
from scipy import integrate

compute_output = widgets.Output()
compute_button = widgets.Button(description="Run time evolution", button_style='danger')

# Helper class for grid attribute
class SimplePotential:
    def __init__(self, values, grid):
        self.values = values
        self.grid = grid
    def __getitem__(self, idx):
        # Supports all numpy indexing (slice, array, int, etc.)
        if isinstance(idx, (int, slice)):
            return np.asarray(self.values)[idx]
        elif isinstance(idx, (np.ndarray, list)):
            return np.asarray(self.values)[np.asarray(idx)]
        else:
            raise TypeError(f"Unsupported index type: {type(idx)}")
    def __array__(self):
        return np.asarray(self.values)

def run_time_evolution(b):
    with compute_output:
        clear_output(wait=True)
        if not selected_quasiparticles:
            print("No quasiparticle selected for Compute run.")
            return
        # Read parameters
        try:
            uxgrid_num = int(loaded_params.get('uxgrid_num', 4096))
            uxgrid_dx = float(loaded_params.get('uxgrid_dx', 0.025))
            uxgrid_width = uxgrid_num * uxgrid_dx
            uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)
            # Laser parameters (from Laser tab)
            spot_size_nm = laser_spot_size_widget.value
            field_strength_gvm = laser_field_strength_widget.value
            central_wavelength_nm = laser_wavelength_widget.value
            num_of_cycles = laser_cycles_widget.value
            beta = laser_beta_widget.value
            # Time grid
            t_start = 0
            t_stop = num_of_cycles * 2 * np.pi / (3e8 / (central_wavelength_nm * 1e-9))
            utgrid_dt = 0.05
            utgrid = np.arange(t_start, t_stop, utgrid_dt)
            # Model potential (simplified)
            wallwidth = float(loaded_params.get('wallwidth', 3.0))
            wallheight = float(loaded_params.get('wallheight', 1.0))
            wallrise = float(loaded_params.get('wallrise', 1.0))
            wellwidth = float(loaded_params.get('wellwidth', 10.0))
            welldepth = float(loaded_params.get('welldepth', 1.0))
            wellfall = float(loaded_params.get('wellfall', 1.0))
            num_cells = int(loaded_params.get('num_cells', 1))
            cell_spacing = float(loaded_params.get('cell_spacing', 0.0))
            pot = np.zeros_like(uxgrid)
            cell_length = 2*wallwidth + wellwidth + cell_spacing
            for i in range(num_cells):
                offset = (i - (num_cells-1)/2) * cell_length
                left = offset - (wellwidth/2 + wallwidth)
                right = offset + (wellwidth/2 + wallwidth)
                well_left = offset - wellwidth/2
                well_right = offset + wellwidth/2
                pot[(uxgrid >= left) & (uxgrid < well_left)] = wallheight
                pot[(uxgrid > well_right) & (uxgrid <= right)] = wallheight
                pot[(uxgrid >= well_left) & (uxgrid <= well_right)] = -welldepth
            modelpot = SimplePotential(pot, uxgrid)
            from quantum_toolkit import potentials as pots
            s_min = -0.5 * spot_size_nm
            s_max = 0.5 * spot_size_nm
            t_max = t_stop
            laserpot = pots.SmoothLaserPotential(
                uxgrid,
                spatial_min=s_min, spatial_max=s_max,
                temporal_max=t_max, temporal_min=t_start,
                noc=num_of_cycles,
                amplitude=field_strength_gvm,
                beta=beta
            )
            laser_potentials = [laserpot]
        except Exception as e:
            print("Parameter read error:", e)
            return

        # Output HDF5 file
        h5_path = get_hdf5_path()
        try:
            h5f = h5py.File(h5_path, "a")
        except Exception as e:
            print(f"Error opening HDF5 file: {e}")
            return

        run_num = 1
        for qp in selected_quasiparticles:
            kin = qp["wavenumber"]
            ein = qp["energy"]
            psi0 = qp.get("state", None)
            if psi0 is None:
                print(f"Quasiparticle is missing wavefunction, skipping.")
                continue
            for lp in laser_potentials:
                print(f"Run {run_num}:")
                print(f"- energy: {ein}")
                print(f"- wavenumber: {kin}")
                print(f"- pulse width: {lp.temporal_width}")
                print(f"- pulse omega0: {getattr(lp, 'omega0', 'n.a.')}")
                try:
                    psite = quat.SplitTimeEvolutionCalculator(
                        psi0_omega=psi0["angular_frequency"],#ein/1.0,  # hbar=1
                        psi0_initial=psi0,
                        scalarpot=modelpot,
                        vectorpot=lp,
                        dt=utgrid_dt, t_start=t_start, t_stop=t_stop
                    )
                    psite.run()
                except Exception as e:
                    print(f"Error during run: {e}")
                    continue
                grpname = f"compute/run_{run_num}"
                if grpname in h5f:
                    del h5f[grpname]
                grp = h5f.create_group(grpname)
                grp.attrs["energy"] = ein
                grp.attrs["wavenumber"] = kin
                grp.create_dataset("psi_time_evolution", data=psite.psi1time_evolution)
                grp.create_dataset("uxgrid", data=uxgrid)
                grp.create_dataset("utgrid", data=utgrid)
                # Plot: probability density
                X, Y = np.meshgrid(uxgrid, utgrid)
                Z = np.transpose(quat.probability_density(psite.psi1time_evolution))
                plt.figure(figsize=(6,3))
                pcm = plt.pcolormesh(X, Y, Z, cmap='bwr')
                plt.colorbar(pcm)
                plt.title(f'Prob. dens. (run {run_num})')
                plt.xlabel("space")
                plt.ylabel("time")
                plt.xlim(-15, 15)
                plt.ylim(0, 2*lp.temporal_width)
                plt.show()
                # Current plot
                pc = quat.probability_current(psite.psitimeevolution)
                left_index = 0
                right_index = -1
                pc_left = pc[left_index, :]
                pc_right = pc[right_index, :]
                plt.figure()
                plt.plot(utgrid, pc_left, label="current left")
                plt.plot(utgrid, pc_right, label="current right")
                plt.xlabel("time")
                plt.ylabel("current")
                plt.legend()
                plt.grid()
                plt.show()
                # Charge plot
                charge_left = integrate.cumtrapz(pc_left, utgrid, initial=0)
                charge_right = integrate.cumtrapz(pc_right, utgrid, initial=0)
                plt.figure()
                plt.plot(utgrid, charge_left, label="charge left")
                plt.plot(utgrid, charge_right, label="charge right")
                plt.xlabel("time")
                plt.ylabel("charge")
                plt.legend()
                plt.grid()
                plt.show()
                run_num += 1
        h5f.close()
        print("Compute run finished, results saved to HDF5 file.")

compute_button.on_click(run_time_evolution)

compute_tab = widgets.VBox([
    widgets.Label("Time evolution for selected quasiparticles (Compute):"),
    compute_button,
    compute_output
])

## 8. GUI Assembly

All tabs in one place, for easy access.

In [8]:
# Assemble tabs
tabs = widgets.Tab(children=[
    file_tab,
    param_tab,
    mask_tab,
    quasiparticle_tab,
    laser_tab,
    vcap_tab,
    compute_tab
])
tab_titles = [
    'File operations', 'Parameters', 'Mask', 'Quasi Particles', 'Laser', 'VCAP', 'Compute'
]
for i, title in enumerate(tab_titles):
    tabs.set_title(i, title)

display(tabs)

In [9]:
print(selected_quasiparticles[0].keys())

IndexError: list index out of range